# 012. 머신러닝 입력변수 설명

GTX-A 학습 및 GTX-B·C 적용에 사용하는 입력변수의 정의와 구축 기준을 정리한다. 우선 **GTX 경쟁 목적지 비율**을 정리한다.

## 1. GTX 경쟁 목적지 비율

이 변수는 특정 노선 전체를 경쟁노선으로 지정한 더미변수가 아니다. 각 격자에서 출발한 서울행 승객 가운데 실제 하차 정류장이 GTX-A의 서울 측 주요 목적지인 **연신내역 또는 서울역 반경 1km 이내**에 위치한 승객의 비율이다.

\[
GTX\ 경쟁\ 목적지\ 비율(\%)=

rac{연신내역·서울역\ 1km\ 이내\ 하차\ 승객}
{하차좌표가\ 확인된\ 서울행\ 승객}
	imes100
\]

In [ ]:
# 프로젝트 루트(UROP) 자동 탐색 — 절대경로 없이 어느 컴퓨터에서도 동작
import sys
from pathlib import Path
UROP_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.py').is_file() and (p / 'code').is_dir())
for _d in [UROP_ROOT, *sorted(d for d in (UROP_ROOT / 'code').rglob('*') if d.is_dir() and d.name != '__pycache__')]:
    if str(_d) not in sys.path:
        sys.path.append(str(_d))

from pathlib import Path
import pandas as pd
from IPython.display import display

# 저장소 루트 또는 analysis 폴더에서 실행하는 경우 모두 지원
_candidates = [
    UROP_ROOT / "data" / "processed" / "gtx_a" / "05_GTX_경쟁목적지비율_종일_셀노선.csv",
    Path.cwd() / "../data/processed/05_GTX_경쟁목적지비율_종일_셀노선.csv",
    Path.cwd() / "gtx_a_bus_analysis/data/processed/05_GTX_경쟁목적지비율_종일_셀노선.csv",
]
SOURCE = next((p.resolve() for p in _candidates if p.exists()), None)
if SOURCE is None:
    raise FileNotFoundError("05_GTX_경쟁목적지비율_종일_셀노선.csv를 찾을 수 없습니다.")

definition = pd.DataFrame([
    ["변수명", "GTX 경쟁 목적지 비율(%)"],
    ["자료 기준", "2024년 10월 17일 종일 교통카드 합성자료"],
    ["분석 단위", "250m 격자(셀)"],
    ["경쟁 목적지", "연신내역 또는 서울역 중심 반경 1km 이내 하차 정류장"],
    ["분자", "격자에서 승차하여 경쟁 목적지에 하차한 승객 수"],
    ["분모", "격자에서 승차한 서울행 승객 중 하차 좌표가 확인된 승객 수"],
    ["산식", "(GTX 경쟁 목적지 승객 수 ÷ 목적지 분류 가능 서울행 승객 수) × 100"],
    ["자료형", "연속형 변수(0~100%)"],
    ["해석", "값이 높을수록 기존 광역버스 수요 중 GTX-A와 목적지가 중복되는 수요 비중이 큼"],
], columns=["구분", "내용"])

display(definition.style.hide(axis="index"))

구분,내용
변수명,GTX 경쟁 목적지 비율(%)
자료 기준,2024년 10월 17일 종일 교통카드 합성자료
분석 단위,250m 격자(셀)
경쟁 목적지,연신내역 또는 서울역 중심 반경 1km 이내 하차 정류장
분자,격자에서 승차하여 경쟁 목적지에 하차한 승객 수
분모,격자에서 승차한 서울행 승객 중 하차 좌표가 확인된 승객 수
산식,(GTX 경쟁 목적지 승객 수 ÷ 목적지 분류 가능 서울행 승객 수) × 100
자료형,연속형 변수(0~100%)
해석,값이 높을수록 기존 광역버스 수요 중 GTX-A와 목적지가 중복되는 수요 비중이 큼


### 경쟁 목적지 승객이 확인된 노선

전체 분석대상 24개 노선 중 연신내역·서울역 반경 1km 이내 하차 승객이 1명 이상 확인된 노선이다. 아래 노선 전체를 일괄적으로 경쟁노선으로 처리한 것이 아니라, 최종 머신러닝 변수는 **각 격자의 승객 비율**로 계산했다.

In [ ]:
raw = pd.read_csv(SOURCE, encoding="utf-8-sig", dtype={"노선": str})
CLASSIFIABLE = "2024년 종일 목적지 분류 가능 승차인원"
COMPETING = "2024년 종일 GTX 경쟁 목적지 승차인원"

route_table = raw.groupby("노선", as_index=False)[[CLASSIFIABLE, COMPETING]].sum()
route_table["GTX 경쟁 목적지 승객 비율(%)"] = (
    route_table[COMPETING] / route_table[CLASSIFIABLE].replace(0, pd.NA) * 100
)
route_table = route_table[route_table[COMPETING] > 0].copy()
route_table = route_table.sort_values(COMPETING, ascending=False).reset_index(drop=True)
route_table.index = route_table.index + 1
route_table.index.name = "순번"
route_table = route_table.rename(columns={
    CLASSIFIABLE: "목적지 분류 가능 승객(명)",
    COMPETING: "GTX 경쟁 목적지 승객(명)",
})
route_table["GTX 경쟁 목적지 승객 비율(%)"] = route_table["GTX 경쟁 목적지 승객 비율(%)"].round(1)

display(route_table)
print(f"전체 분석 노선: {raw['노선'].nunique()}개")
print(f"경쟁 목적지 승객이 확인된 노선: {len(route_table)}개")

,노선,목적지 분류 가능 승객(명),GTX 경쟁 목적지 승객(명),GTX 경쟁 목적지 승객 비율(%)
순번,,,,
1,1000,3889,1074.0,27.6
2,1200,1797,404.0,22.5
3,M7111,1124,309.0,27.5
4,M7154,369,209.0,56.6
5,9710,240,128.0,53.3
6,9709,164,100.0,61.0
7,1900,114,47.0,41.2
8,9710-1,18,9.0,50.0
9,1100,18,2.0,11.1


전체 분석 노선: 24개
경쟁 목적지 승객이 확인된 노선: 9개


### 해석 시 주의사항

- `경쟁 목적지 비율 50%`는 해당 노선이나 격자의 승차수요가 GTX 개통 후 50% 감소한다는 뜻이 아니다.
- 이 값은 개통 전 광역버스 이용수요 가운데 GTX-A의 주요 서울 측 목적지와 공간적으로 중복되는 수요의 비중이다.
- GTX-B·C 적용 시에는 연신내역·서울역을 그대로 사용하지 않고, 각 노선의 GTX 정차역 및 주요 서울 도착지를 기준으로 공통 정의를 다시 설정해야 한다.

## 2. 경쟁 목적지 이용노선의 2025년 변화

2024년에 경쟁 목적지 승객이 1명 이상 확인된 9개 노선을 고정하여, 2025년 11월 6일의 서울행 및 경쟁 목적지 수요를 비교한다. 두 연도 모두 공급구조가 유지된 동일 126개 격자와 연신내역·서울역 반경 1km 기준을 적용한다.

In [ ]:
# 2024년 경쟁 목적지 이용 9개 노선의 2025년 변화
import sys
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = UROP_ROOT
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import config

ROUTES_2024 = ["1000", "1200", "M7111", "M7154", "9710", "9709", "1900", "9710-1", "1100"]
pairs = config.read_csv(config.CELL_ROUTE_ALL_DAY_PATH)
pairs["셀 ID"] = pd.to_numeric(pairs["셀 ID"], errors="coerce").astype("Int64")
pairs["노선"] = pairs["노선"].astype(str).str.removesuffix(".0")
pairs = pairs[pd.to_numeric(pairs["y2024"], errors="coerce").fillna(0) > 0][["셀 ID", "노선"]]

raw_paths = {
    2024: PROJECT / "data/raw/gtx_a/transport_card/gtx_a_transport_card_20241017_raw_with_coords_api_filled.csv",
    2025: PROJECT / "data/raw/gtx_a/transport_card/gtx_a_transport_card_20251106_raw_with_coords.csv",
}
stop_map = config.stop_to_cell_map()
annual_rows = []
for year, path in raw_paths.items():
    raw_year = config.read_csv(path)
    trips = pd.DataFrame({
        "셀 ID": raw_year["승차정류장ID"].astype(str).str.removesuffix(".0").map(stop_map),
        "노선": raw_year["query_route_no"].astype(str).str.removesuffix(".0"),
        "서울 하차": pd.to_numeric(raw_year["goff_ctpv_cd"], errors="coerce").eq(config.SEOUL),
        "하차위도": pd.to_numeric(raw_year["하차위도"], errors="coerce"),
        "하차경도": pd.to_numeric(raw_year["하차경도"], errors="coerce"),
        "승차인원": pd.to_numeric(raw_year["utztn_nope"], errors="coerce").fillna(0),
    }).dropna(subset=["셀 ID"])
    trips["셀 ID"] = trips["셀 ID"].astype("Int64")
    trips = trips.merge(pairs, on=["셀 ID", "노선"], how="inner")
    trips = trips[trips["서울 하차"]].copy()
    trips["분류가능"] = trips[["하차위도", "하차경도"]].notna().all(axis=1)
    trips["경쟁목적지"] = False
    idx = trips["분류가능"]
    trips.loc[idx, "경쟁목적지"] = config.is_competing_destination(
        trips.loc[idx, "하차위도"], trips.loc[idx, "하차경도"]
    )
    for route in ROUTES_2024:
        part = trips[trips["노선"] == route]
        annual_rows.append({
            "노선": route,
            "연도": year,
            "서울행 승객": int(part["승차인원"].sum()),
            "분류 가능 승객": int(part.loc[part["분류가능"], "승차인원"].sum()),
            "경쟁 목적지 승객": int(part.loc[part["경쟁목적지"], "승차인원"].sum()),
        })

annual = pd.DataFrame(annual_rows)
wide = annual.pivot(index="노선", columns="연도", values=["서울행 승객", "분류 가능 승객", "경쟁 목적지 승객"])
comparison_rows = []
for route in ROUTES_2024:
    a = wide.loc[route]
    c24, c25 = int(a[("경쟁 목적지 승객", 2024)]), int(a[("경쟁 목적지 승객", 2025)])
    d24, d25 = int(a[("분류 가능 승객", 2024)]), int(a[("분류 가능 승객", 2025)])
    comparison_rows.append({
        "노선": route,
        "경쟁 승객 2024(명)": c24,
        "경쟁 승객 2025(명)": c25,
        "경쟁 승객 변화율(%)": round((c25-c24)/c24*100, 1) if c24 else np.nan,
        "경쟁 비율 2024(%)": round(c24/d24*100, 1) if d24 else np.nan,
        "경쟁 비율 2025(%)": round(c25/d25*100, 1) if d25 else np.nan,
    })
comparison = pd.DataFrame(comparison_rows)

def total(year, column):
    return int(annual.loc[annual["연도"] == year, column].sum())

seoul24, seoul25 = total(2024, "서울행 승객"), total(2025, "서울행 승객")
comp24, comp25 = total(2024, "경쟁 목적지 승객"), total(2025, "경쟁 목적지 승객")
class24, class25 = total(2024, "분류 가능 승객"), total(2025, "분류 가능 승객")
summary = pd.DataFrame([
    ["서울행 승객", seoul24, seoul25, round((seoul25-seoul24)/seoul24*100, 1), "명"],
    ["경쟁 목적지 승객", comp24, comp25, round((comp25-comp24)/comp24*100, 1), "명"],
    ["경쟁 목적지 비율", round(comp24/class24*100, 1), round(comp25/class25*100, 1), round(comp25/class25*100-comp24/class24*100, 1), "%, 변화는 %p"],
], columns=["지표", "2024년", "2025년", "변화", "단위"])

display(comparison)
display(summary)


노선,경쟁 승객 2024(명),경쟁 승객 2025(명),경쟁 승객 변화율(%),경쟁 비율 2024(%),경쟁 비율 2025(%)
1000,1074,201,-81.3,27.6,18.7
1200,404,107,-73.5,22.5,20.5
M7111,309,27,-91.3,27.5,17.2
M7154,209,23,-89.0,56.6,30.7
9710,128,37,-71.1,53.3,52.9
9709,100,54,-46.0,61.0,65.1
1900,47,12,-74.5,41.2,40.0
9710-1,9,2,-77.8,50.0,33.3
1100,2,0,-100.0,11.1,0.0


지표,2024년,2025년,변화,단위
서울행 승객,7733.0,2024.0,-73.8,명
경쟁 목적지 승객,2282.0,463.0,-79.7,명
경쟁 목적지 비율,29.5,22.9,-6.6,"%, 변화는 %p"


### 결과 해석

- 9개 노선의 서울행 승객은 7,733명에서 2,024명으로 73.8% 감소했다.
- 경쟁 목적지 승객은 2,282명에서 463명으로 79.7% 감소하여 서울행 전체 수요보다 감소폭이 컸다.
- 경쟁 목적지 비율은 29.5%에서 22.9%로 6.6%p 낮아졌다.
- `9709`는 경쟁 승객 수가 감소했지만 경쟁 비율은 61.0%에서 65.1%로 높아졌고, `9710`은 53% 수준을 유지했다.
- `M7111`과 `M7154`의 경쟁 목적지 승객은 약 90% 감소했으며, `1100`은 2025년 경쟁 목적지 승객이 0명으로 나타났다.
- 이 결과는 하루 합성 교통카드 자료의 승객 변화이며, 노선 폐선이나 실제 운행횟수 감소를 직접 의미하지 않는다.